# Description

In this notebook, we benchmark ComplexEQL algorithm on the Korns benchmarks.

In [ ]:
# ============================================================
# Run Korns benchmarks with ComplexEQL using your UPDATED config
# ============================================================

from dataclasses import dataclass
from typing import Optional
import numpy as np
import sympy as sp

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from config.korns_config import BENCH, FEATURE_NAMES, CEQL_TRAIN, CEQL
from src.korns_core import RunConfig, load_korns_hdf5, run_benchmark, SRFitResult

from src.ComplexEQL import ComplexEQL            # adjust if your module path differs
from src.utils import set_seed, train


@dataclass
class ComplexEQLKornsRegressor:
    name: str = "complexeql"
    device: Optional[str] = None  # override; else uses CEQL_TRAIN.device
    seed: int = 42
    rounding_decimals: int = 2

    def fit_predict(self, X_train, y_train, X_test) -> SRFitResult:
        mcfg = CEQL_TRAIN
        ncfg = CEQL

        # Korns has 5 inputs; keep config consistent with data
        ncfg.n_input_fields = int(X_train.shape[1])

        device_str = self.device if self.device is not None else getattr(mcfg, "device", "cpu")
        device = torch.device(device_str)

        set_seed(self.seed)

        Xtr = torch.tensor(np.asarray(X_train, dtype=np.float32), device=device)
        ytr = torch.tensor(np.asarray(y_train, dtype=np.float32).reshape(-1, 1), device=device)

        dataset = TensorDataset(Xtr, ytr)
        dataloader = DataLoader(
            dataset,
            batch_size=int(getattr(mcfg, "train_batch_size", 2**14)),
            shuffle=True,
            drop_last=False,
        )

        model = ComplexEQL(ncfg).to(device)
        loss_fn = nn.MSELoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=float(getattr(mcfg, "lr", 1e-3)))

        scheduler = None
        if getattr(mcfg, "scheduler", None) == "ReduceLROnPlateau":
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                optimizer, **getattr(mcfg, "schedulerparams", {})
            )
        elif getattr(mcfg, "scheduler", None) is None:
            scheduler = None
        else:
            raise ValueError(f"Unknown scheduler: {mcfg.scheduler}")

        model, _history = train(
            model=model,
            dataloader=dataloader,
            optimizer=optimizer,
            loss_fn=loss_fn,
            cfg=mcfg,
            device=device,
            scheduler=scheduler,
        )

        model.eval()
        with torch.no_grad():
            y_pred_train = model(Xtr, op_params=None).real.squeeze(-1).detach().cpu().numpy()

            Xte = torch.tensor(np.asarray(X_test, dtype=np.float32), device=device)
            y_pred_test = model(Xte, op_params=None).real.squeeze(-1).detach().cpu().numpy()

        try:
            syms = [sp.Symbol(n) for n in FEATURE_NAMES[: ncfg.n_input_fields]]
            expr = model.get_symbolic_expression(syms, rounding_decimals=int(self.rounding_decimals))
        except Exception:
            expr = None

        return SRFitResult(
            expr=expr,
            y_pred_train=np.asarray(y_pred_train, dtype=np.float64).reshape(-1),
            y_pred_test=np.asarray(y_pred_test, dtype=np.float64).reshape(-1),
            metadata=None,
        )


# ============================================================
# Benchmark run
# ============================================================

cfg = RunConfig(
    hdf5_path=BENCH.hdf5_path,
    test_size=BENCH.test_size,
    split_seed=BENCH.split_seed,
    per_problem_seed_offset=BENCH.per_problem_seed_offset,
    algo_seed_offset=BENCH.algo_seed_offset,
    run_seed_offset=BENCH.run_seed_offset,
)

datasets = load_korns_hdf5(cfg.hdf5_path)
del datasets["P4"], datasets["P11"], datasets["P12"], datasets["P13"], datasets["P14"], datasets["P15"]

rows = run_benchmark(
    datasets=datasets,
    algorithms=[ComplexEQLKornsRegressor(name="complexeql", seed=42)],
    config=cfg,
    n_runs=BENCH.n_runs,
    feature_names=FEATURE_NAMES,
    results_csv_path="korns_complexeql_benchmark_results.csv",
)
